# Thực nghiệm đọc đồng hồ nước — Phúc

Notebook tổng hợp E1–E4. E1/E2 do Phúc thực hiện; E3/E4 đọc sản phẩm của Quang khi được cung cấp. Chưa có dữ liệu thật trên máy: notebook báo rõ kết quả thiếu, không thay bằng số giả.

Chạy hai lệnh tune/evaluate trong `experiments/phuc/README.md` trước khi phân tích. Restart & Run All có thể chạy khi chưa có kết quả; điều đó chỉ xác nhận notebook hoạt động, chưa xác nhận thực nghiệm hoàn tất.

In [1]:
from pathlib import Path
import csv, json
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
RESULTS = ROOT / 'experiments/phuc/results'
def table(name):
    path = RESULTS / name
    if not path.exists():
        display(Markdown(f'**Chưa có `{name}` — chưa thể kết luận.**'))
        return []
    with path.open() as f:
        rows = list(csv.DictReader(f))
    display(rows)
    return rows


## E1 — Chứng cứ thách thức

So sánh đúng cùng vị trí chữ số qua ba tầng. Quan sát bốn số đầu làm đối chứng. Độ sáng, tương phản và Laplacian variance chỉ là proxy của nhiễu; không thay thế kiểm soát loại đồng hồ. Quan hệ thống kê không đủ kết luận độ lăn là nguyên nhân duy nhất.

In [2]:
summary = table('summary.csv')
controls = table('e1_control_summary.csv')

**Chưa có `summary.csv` — chưa thể kết luận.**

**Chưa có `e1_control_summary.csv` — chưa thể kết luận.**

In [3]:
if summary:
    tiers = ['light', 'medium', 'heavy']
    for method in sorted({r['method'] for r in summary}):
        group = {r['tier']: r for r in summary if r['method'] == method}
        plt.plot(tiers, [float(group[t]['d5']) for t in tiers], marker='o', label=method)
    plt.ylim(0,1); plt.ylabel('Last digit accuracy'); plt.legend(); plt.show()
else:
    print('Chưa có kết quả E1 để nhận xét mức suy giảm.')

Chưa có kết quả E1 để nhận xét mức suy giảm.


## E2 — Khảo sát tham số trên dev_val

Pitch điều chỉnh khoảng cách giữa hai chữ số; bán kính dịch ngang bù lệch ô; ngưỡng quyết định thời điểm chuyển nhãn. Mỗi tham số có ít nhất ba giá trị. Bảng kèm accuracy theo tầng; ảnh trung gian dùng mẫu dev_val, không chọn ảnh eval để điều chỉnh phương pháp.

In [4]:
grid = table('e2_grid.csv')
if (RESULTS/'selected.json').exists():
    display(json.loads((RESULTS/'selected.json').read_text()))
if (RESULTS/'e2_intermediates.png').exists():
    display(Image(filename=str(RESULTS/'e2_intermediates.png')))

**Chưa có `e2_grid.csv` — chưa thể kết luận.**

In [5]:
if grid:
    best=max(grid,key=lambda r:float(r['last_accuracy']))
    display(Markdown(f"Cấu hình tốt nhất trên dev_val: pitch={best['pitch']}, shift={best['max_shift']}, threshold={best['threshold']}; accuracy={float(best['last_accuracy']):.3f}. Chưa được suy ra cải thiện trên eval từ bảng này."))
    for key in ('pitch','max_shift','threshold'):
        xs=sorted({float(r[key]) for r in grid})
        # Marginal mean across other settings, not isolated causal effects.
        ys=[sum(float(r['last_accuracy']) for r in grid if float(r[key])==x)/sum(float(r[key])==x for r in grid) for x in xs]
        plt.plot(xs,ys,marker='o'); plt.xlabel(key); plt.ylabel('Mean dev_val accuracy across other settings'); plt.show()

## E3 — Loại bỏ thành phần (Quang)

Đầu vào dự kiến: `e3_summary.csv`, cùng schema bảng summary. So sánh enabled=True/False với cùng mẫu, tiền xử lý, max_shift và tập đánh giá. Kiểm thử mã đã kiểm tra disabled tương đương baseline cùng cấu hình; kết quả ảnh thật cần được Quang bổ sung.

In [6]:
e3 = table('e3_summary.csv')

**Chưa có `e3_summary.csv` — chưa thể kết luận.**

## E4 — Điểm gãy (Quang)

Đầu vào dự kiến: `e4_summary.csv`. Đối chiếu các dự đoán đã commit trong PR #22 trước cài đặt: quét pha tổng hợp, lỗi lân cận ngưỡng nhãn, sai lệch bước bánh xe. Chưa có kết quả thì không xác nhận dự đoán. Phải nêu riêng miền tổng hợp và dữ liệu thật.

In [7]:
e4 = table('e4_summary.csv')

**Chưa có `e4_summary.csv` — chưa thể kết luận.**

## Trạng thái và nhận xét

- #3 kế thừa PR #21, không ghi nhận là một lần tái lập mới.
- Mã #8 và pipeline #6/#11 đã có kiểm thử; hiệu quả cần ảnh thật.
- Không tinh chỉnh tham số theo eval. Nếu thay đổi sau khi xem eval, phải công bố là vòng phát triển tiếp theo.
- Notebook chưa hoàn tất nội dung thực nghiệm E1–E4 cho tới khi có bảng kết quả và nhận xét dựa trên dữ liệu.